<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:

using System;
using System.Collections.Generic;
using System.Linq;

// интерфейсы

public interface ISearchable
{
    List<Item> SearchByName(string keyword);
}

public interface IReportable
{
    string GenerateReport();
}

public interface IInventoryOperations
{
    bool AddItem(Item item);
    bool RemoveItem(Item item);
    string GenerateReport();
    void PrintSummary();
}

// товар

public class Item
{
    public string Name { get; private set; }
    public int Volume { get; private set; }
    public string Category { get; private set; }
    public double Weight { get; private set; }
    public double Price { get; private set; }
    public bool IsFragile { get; private set; }
    public DateTime AddedDate { get; private set; }

    public Item(string name, int volume, string category = "Общая",
        double weight = 0, double price = 0, bool isFragile = false)
    {
        Name = name;
        Volume = volume;
        Category = category;
        Weight = weight;
        Price = price;
        IsFragile = isFragile;
        AddedDate = DateTime.Now;
    }

    public double GetTotalValue(int quantity) => Price * quantity;

    public string CheckCondition() => IsFragile ? "Хрупкий" : "Обычный";

    public override string ToString()
    {
        return $"[Товар: {Name}, Объем: {Volume}, Категория: {Category}, Цена: {Price}]";
    }
}

// основной класс склада

public abstract class Inventory : ISearchable, IReportable, IInventoryOperations
{
    public int WarehouseId { get; private set; }
    public string WarehouseName { get; private set; }
    public int StorageCapacity { get; private set; }
    public string Location { get; protected set; }
    public bool IsOpen { get; protected set; }
    public string ManagerName { get; protected set; }
    public int MaxItemsCount { get; protected set; }

    public DateTime CreatedAt { get; private set; }
    public DateTime LastInventoryCheck { get; protected set; }
    public bool IsUnderMaintenance { get; protected set; }

    protected List<Item> Items { get; private set; }

    public int CurrentVolume => Items.Sum(item => item.Volume);

    protected Inventory(int warehouseId, string warehouseName, int storageCapacity)
    {
        WarehouseId = warehouseId;
        WarehouseName = warehouseName;
        StorageCapacity = storageCapacity;
        Items = new List<Item>();

        Location = "Не указан";
        IsOpen = true;
        ManagerName = "Не назначен";
        MaxItemsCount = 1000;

        CreatedAt = DateTime.Now;
        LastInventoryCheck = DateTime.Now;
        IsUnderMaintenance = false;
    }

    public virtual bool CanAddItem(Item item)
    {
        return CurrentVolume + item.Volume <= StorageCapacity &&
               Items.Count < MaxItemsCount;
    }

    public virtual bool AddItem(Item item)
    {
        if (!IsOpen)
        {
            Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: склад закрыт.");
            return false;
        }

        if (CanAddItem(item))
        {
            Items.Add(item);
            Console.WriteLine($"[Склад '{WarehouseName}'] Товар '{item.Name}' успешно добавлен.");
            return true;
        }

        Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: Недостаточно места для товара '{item.Name}'.");
        return false;
    }

    public virtual bool RemoveItem(Item item)
    {
        if (Items.Contains(item))
        {
            Items.Remove(item);
            Console.WriteLine($"[Склад '{WarehouseName}'] Товар '{item.Name}' успешно удален.");
            return true;
        }

        Console.WriteLine($"[Склад '{WarehouseName}'] Ошибка: Товар '{item.Name}' не найден на складе.");
        return false;
    }

    public virtual bool TransferItem(Item item, Inventory targetWarehouse)
    {
        if (Items.Contains(item) && targetWarehouse.CanAddItem(item))
        {
            RemoveItem(item);
            targetWarehouse.AddItem(item);

            Console.WriteLine(
                $"[Трансфер] Товар '{item.Name}' успешно перемещен со склада '{WarehouseName}' на '{targetWarehouse.WarehouseName}'.");

            return true;
        }

        Console.WriteLine(
            $"[Трансфер] Не удалось переместить '{item.Name}'. Проверьте наличие товара и вместимость склада '{targetWarehouse.WarehouseName}'.");

        return false;
    }

    public List<Item> GetItemsByCategory(string category)
    {
        return Items
            .Where(i => i.Category.Equals(category, StringComparison.OrdinalIgnoreCase))
            .ToList();
    }

    public List<Item> SearchByName(string keyword)
    {
        return Items
            .Where(i => i.Name.Contains(keyword, StringComparison.OrdinalIgnoreCase))
            .ToList();
    }

    public List<Item> SearchByName(string keyword, bool caseSensitive)
    {
        return Items
            .Where(item =>
                caseSensitive
                    ? item.Name.Contains(keyword)
                    : item.Name.Contains(keyword, StringComparison.OrdinalIgnoreCase))
            .ToList();
    }

    public List<Item> SearchByName(string keyword, string category)
    {
        return Items
            .Where(item =>
                item.Name.Contains(keyword, StringComparison.OrdinalIgnoreCase) &&
                item.Category.Equals(category, StringComparison.OrdinalIgnoreCase))
            .ToList();
    }

    public virtual string GenerateReport()
    {
        return $"Склад '{WarehouseName}' (ID: {WarehouseId}): позиций {Items.Count}, объем {CurrentVolume}/{StorageCapacity}, открыт: {IsOpen}";
    }

    public virtual void GetStorageStatus()
    {
        int freeSpace = StorageCapacity - CurrentVolume;

        Console.WriteLine($"--- Статус склада '{WarehouseName}' (ID: {WarehouseId}) ---");
        Console.WriteLine($"Вместимость: {StorageCapacity} | Занято: {CurrentVolume} | Свободно: {freeSpace}");
        Console.WriteLine($"Локация: {Location} | Открыт: {IsOpen} | Управляющий: {ManagerName}");
    }

    public void Close()
    {
        IsOpen = false;
        Console.WriteLine($"Склад '{WarehouseName}' закрыт.");
    }

    public void Open()
    {
        IsOpen = true;
        Console.WriteLine($"Склад '{WarehouseName}' открыт.");
    }

    // новые методы

    public int GetFreeSpace()
    {
        return StorageCapacity - CurrentVolume;
    }

    public void UpdateInventoryCheck()
    {
        LastInventoryCheck = DateTime.Now;
        Console.WriteLine($"Проверка склада '{WarehouseName}' обновлена.");
    }

    public void SetMaintenance(bool status)
    {
        IsUnderMaintenance = status;
        Console.WriteLine($"Обслуживание склада '{WarehouseName}': {status}.");
    }

    // реализация интерфейса

    void IInventoryOperations.PrintSummary()
    {
        GetStorageStatus();
    }
}

// личный склад

public class PersonalInventory : Inventory
{
    public string OwnerName { get; private set; }
    public string SecurityCode { get; private set; }
    public bool IsLocked { get; private set; }
    public int MaxItemWeight { get; private set; }

    public string AccessLevel { get; private set; }
    public DateTime LastAccessDate { get; private set; }
    public int AccessAttempts { get; private set; }

    public PersonalInventory(int warehouseId, string warehouseName,
        int storageCapacity, string ownerName)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        OwnerName = ownerName;
        SecurityCode = "0000";
        IsLocked = false;
        MaxItemWeight = 50;

        Location = "Дом";
        ManagerName = ownerName;

        AccessLevel = "Владелец";
        LastAccessDate = DateTime.Now;
        AccessAttempts = 0;
    }

    public bool Lock(string code)
    {
        if (code == SecurityCode)
        {
            IsLocked = true;
            Console.WriteLine($"Склад '{WarehouseName}' заблокирован.");
            return true;
        }

        Console.WriteLine("Неверный код!");
        return false;
    }

    public bool Unlock(string code)
    {
        if (code == SecurityCode)
        {
            IsLocked = false;
            Console.WriteLine($"Склад '{WarehouseName}' разблокирован.");
            return true;
        }

        Console.WriteLine("Неверный код!");
        return false;
    }

    public override bool AddItem(Item item)
    {
        if (IsLocked)
        {
            Console.WriteLine("[Личный склад] Ошибка: склад заблокирован.");
            return false;
        }

        if (item.Weight > MaxItemWeight)
        {
            Console.WriteLine(
                $"[Личный склад] Ошибка: вес товара {item.Weight} кг превышает лимит {MaxItemWeight} кг.");
            return false;
        }

        return base.AddItem(item);
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();

        Console.WriteLine($"Владелец: {OwnerName} | Заблокирован: {IsLocked}");
        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() +
               $", владелец: {OwnerName}, заблокирован: {IsLocked}";
    }

    // новые методы

    public void ChangeAccessLevel(string level)
    {
        AccessLevel = level;
        Console.WriteLine($"Уровень доступа изменен: {level}");
    }

    public void RegisterAccess()
    {
        AccessAttempts++;
        LastAccessDate = DateTime.Now;
    }

    public void ResetAccessAttempts()
    {
        AccessAttempts = 0;
        Console.WriteLine("Счетчик попыток доступа сброшен.");
    }
}

// групповой склад

public class GroupInventory : Inventory
{
    public string ProductGroup { get; private set; }
    public string Department { get; private set; }
    public double Budget { get; private set; }
    public int EmployeeCount { get; private set; }

    public string GroupLeader { get; private set; }
    public DateTime LastAuditDate { get; private set; }
    public int CompletedOrders { get; private set; }

    public GroupInventory(int warehouseId, string warehouseName,
        int storageCapacity, string productGroup)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        ProductGroup = productGroup;
        Department = "Общий";
        Budget = 0;
        EmployeeCount = 0;

        Location = "Офис";

        GroupLeader = "Не назначен";
        LastAuditDate = DateTime.Now;
        CompletedOrders = 0;
    }

    public void SetBudget(double budget)
    {
        Budget = budget;
        Console.WriteLine($"[Групповой склад] Бюджет установлен: {budget} руб.");
    }

    public void AddEmployee()
    {
        EmployeeCount++;
        Console.WriteLine($"[Групповой склад] Сотрудник добавлен. Всего: {EmployeeCount}");
    }

    public override bool AddItem(Item item)
    {
        if (!item.Category.Equals(ProductGroup, StringComparison.OrdinalIgnoreCase))
        {
            Console.WriteLine(
                $"[Групповой склад] Ошибка: Товар '{item.Name}' (Категория: {item.Category}) не соответствует группе склада '{ProductGroup}'.");
            return false;
        }

        if (base.AddItem(item))
        {
            Console.WriteLine(
                $"[Групповой склад] Товар '{item.Name}' успешно отнесен к товарной группе '{ProductGroup}'.");
            return true;
        }

        return false;
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();

        Console.WriteLine(
            $"Группа товаров: {ProductGroup} | Отдел: {Department} | Бюджет: {Budget} руб. | Сотрудников: {EmployeeCount}");

        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() +
               $", группа: {ProductGroup}, отдел: {Department}, бюджет: {Budget}";
    }

    // новые методы

    public void SetGroupLeader(string name)
    {
        GroupLeader = name;
        Console.WriteLine($"Руководитель группы: {name}");
    }

    public void CompleteOrder()
    {
        CompletedOrders++;
        Console.WriteLine($"Выполнено заказов: {CompletedOrders}");
    }

    public void PerformAudit()
    {
        LastAuditDate = DateTime.Now;
        Console.WriteLine("Проведена проверка группового склада.");
    }
}

// автоматический склад

public class AutomatedInventory : Inventory
{
    public string AutomationLevel { get; private set; }
    public string RobotModel { get; private set; }
    public string SoftwareVersion { get; private set; }
    public double Efficiency { get; private set; }

    public int RobotCount { get; private set; }
    public DateTime LastAutomationCheck { get; private set; }
    public bool IsRobotActive { get; private set; }

    public AutomatedInventory(int warehouseId, string warehouseName,
        int storageCapacity, string automationLevel)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        AutomationLevel = automationLevel;
        RobotModel = "Стандартный";
        SoftwareVersion = "1.0.0";
        Efficiency = 90.0;

        Location = "Автоматизированный цех";

        RobotCount = 2;
        LastAutomationCheck = DateTime.Now;
        IsRobotActive = true;
    }

    public void UpdateSoftware(string version)
    {
        SoftwareVersion = version;
        Console.WriteLine(
            $"[Автоматизированный склад] ПО обновлено до версии {version}.");
    }

    public override bool RemoveItem(Item item)
    {
        if (base.RemoveItem(item))
        {
            Console.WriteLine(
                $"[Автоматизированный склад] Уровень автоматизации: {AutomationLevel}. Робот-погрузчик завершил операцию удаления товара '{item.Name}'.");
            return true;
        }

        return false;
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();

        Console.WriteLine(
            $"Автоматизация: {AutomationLevel} | Робот: {RobotModel} | ПО: {SoftwareVersion} | Эффективность: {Efficiency}%");

        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() +
               $", автоматизация: {AutomationLevel}, робот: {RobotModel}, ПО: {SoftwareVersion}";
    }

    // новые методы

    public void AddRobot()
    {
        RobotCount++;
        Console.WriteLine($"Количество роботов: {RobotCount}");
    }

    public void CheckAutomation()
    {
        LastAutomationCheck = DateTime.Now;
        Console.WriteLine("Система автоматизации проверена.");
    }

    public void SetRobotActivity(bool status)
    {
        IsRobotActive = status;
        Console.WriteLine($"Активность роботов: {status}");
    }
}

// умный склад

public class SmartAutomatedInventory : AutomatedInventory
{
    public bool HasClimateControl { get; private set; }
    public double Temperature { get; private set; }
    public double Humidity { get; private set; }
    public DateTime LastInspectionDate { get; private set; }

    public bool IsSensorActive { get; private set; }
    public double CarbonDioxideLevel { get; private set; }
    public DateTime LastSensorCheck { get; private set; }

    public SmartAutomatedInventory(int warehouseId, string warehouseName,
        int storageCapacity, string automationLevel)
        : base(warehouseId, warehouseName, storageCapacity, automationLevel)
    {
        HasClimateControl = true;
        Temperature = 20.0;
        Humidity = 45.0;
        LastInspectionDate = DateTime.Now;

        IsSensorActive = true;
        CarbonDioxideLevel = 400;
        LastSensorCheck = DateTime.Now;
    }

    public void CheckClimate()
    {
        Console.WriteLine(
            $"[Умный склад] Температура: {Temperature}°C, Влажность: {Humidity}%, Климат-контроль: {(HasClimateControl ? "активен" : "отключен")}");
    }

    public void SetTemperature(double temp)
    {
        Temperature = temp;
        Console.WriteLine($"[Умный склад] Температура установлена: {temp}°C");
    }

    public override void GetStorageStatus()
    {
        base.GetStorageStatus();

        Console.WriteLine(
            $"Климат-контроль: {(HasClimateControl ? "да" : "нет")} | Температура: {Temperature}°C | Влажность: {Humidity}%");

        Console.WriteLine("---------------------------------------------");
    }

    public override string GenerateReport()
    {
        return base.GenerateReport() +
               $", климат-контроль: {(HasClimateControl ? "да" : "нет")}, температура: {Temperature}°C";
    }

    // новые методы

    public void CheckSensors()
    {
        LastSensorCheck = DateTime.Now;
        Console.WriteLine("Датчики проверены.");
    }

    public void SetSensorActivity(bool status)
    {
        IsSensorActive = status;
        Console.WriteLine($"Активность датчиков: {status}");
    }

    public void UpdateCarbonDioxide(double level)
    {
        CarbonDioxideLevel = level;
        Console.WriteLine($"Уровень CO2: {level} ppm");
    }
}

// коллекция складов

public class InventoryCollection<T> where T : Inventory
{
    private List<T> warehouses = new List<T>();

    public void Add(T warehouse)
    {
        warehouses.Add(warehouse);
        Console.WriteLine($"Склад '{warehouse.WarehouseName}' добавлен в коллекцию.");
    }

    public void Remove(T warehouse)
    {
        if (warehouses.Remove(warehouse))
            Console.WriteLine($"Склад '{warehouse.WarehouseName}' удален из коллекции.");
        else
            Console.WriteLine("Склад не найден.");
    }

    public void DisplayReports()
    {
        foreach (T warehouse in warehouses)
        {
            Console.WriteLine(warehouse.GenerateReport());
        }
    }

    public List<T> FindWarehousesByItemName(string keyword)
    {
        return warehouses
            .Where(warehouse => warehouse.SearchByName(keyword).Count > 0)
            .ToList();
    }

    public int Count()
    {
        return warehouses.Count;
    }
}

// работа со складами

public class InventoryService
{
    private readonly IInventoryOperations _inventory;

    public InventoryService(IInventoryOperations inventory)
    {
        _inventory = inventory;
    }

    public void ProcessInventory(Item item)
    {
        _inventory.AddItem(item);
        _inventory.PrintSummary();
    }

    public void RemoveFromInventory(Item item)
    {
        _inventory.RemoveItem(item);
    }

    public void ShowReport()
    {
        Console.WriteLine(_inventory.GenerateReport());
    }
}

// основная часть программы

InventoryCollection<PersonalInventory> personalWarehouses =
    new InventoryCollection<PersonalInventory>();

InventoryCollection<GroupInventory> groupWarehouses =
    new InventoryCollection<GroupInventory>();

InventoryCollection<Inventory> allWarehouses =
    new InventoryCollection<Inventory>();

// создаем склады

PersonalInventory personal =
    new PersonalInventory(1, "Личный склад", 500, "Иван");

GroupInventory group =
    new GroupInventory(2, "Склад электроники", 1000, "Электроника");

AutomatedInventory automated =
    new AutomatedInventory(3, "Автоматический склад", 2000, "Высокий");

SmartAutomatedInventory smart =
    new SmartAutomatedInventory(4, "Умный склад", 3000, "Максимальный");

// добавляем склады

personalWarehouses.Add(personal);
groupWarehouses.Add(group);

allWarehouses.Add(personal);
allWarehouses.Add(group);
allWarehouses.Add(automated);
allWarehouses.Add(smart);

Console.WriteLine("\nОтчеты по всем складам:");
allWarehouses.DisplayReports();

Console.WriteLine($"\nКоличество складов: {allWarehouses.Count()}");

Item laptop = new Item("Ноутбук", 50, "Электроника", 2.5, 70000);
group.AddItem(laptop);

Console.WriteLine("\nПоиск по названию:");
List<Item> result1 = group.SearchByName("ноутбук");
foreach (Item item in result1)
    Console.WriteLine(item);

Console.WriteLine("\nПоиск по названию с учетом регистра:");
List<Item> result2 = group.SearchByName("Ноутбук", true);
foreach (Item item in result2)
    Console.WriteLine(item);

Console.WriteLine("\nПоиск по названию и категории:");
List<Item> result3 = group.SearchByName("ноутбук", "Электроника");
foreach (Item item in result3)
    Console.WriteLine(item);

Console.WriteLine("\nДополнительные методы базового класса:");

Console.WriteLine($"Свободное место: {group.GetFreeSpace()}");
group.UpdateInventoryCheck();
group.SetMaintenance(true);


Console.WriteLine("\nДополнительные методы личного склада:");

personal.ChangeAccessLevel("Администратор");
personal.RegisterAccess();
personal.ResetAccessAttempts();


Console.WriteLine("\nДополнительные методы группового склада:");

group.SetGroupLeader("Петр");
group.CompleteOrder();
group.PerformAudit();


Console.WriteLine("\nДополнительные методы автоматизированного склада:");

automated.AddRobot();
automated.CheckAutomation();
automated.SetRobotActivity(false);


Console.WriteLine("\nДополнительные методы умного склада:");

smart.CheckSensors();
smart.SetSensorActivity(false);
smart.UpdateCarbonDioxide(450);


Console.WriteLine("\nДемонстрация Dependency Injection:");

InventoryService personalService = new InventoryService(personal);
InventoryService groupService = new InventoryService(group);
InventoryService automatedService = new InventoryService(automated);

Item phone = new Item("Телефон", 20, "Электроника", 0.5, 50000);

Console.WriteLine("\nРабота с личным складом:");
personalService.ProcessInventory(phone);
personalService.ShowReport();

Console.WriteLine("\nРабота с групповым складом:");
groupService.ProcessInventory(phone);
groupService.ShowReport();

Console.WriteLine("\nРабота с автоматизированным складом:");
automatedService.ProcessInventory(phone);
automatedService.ShowReport();


Console.WriteLine("\nЯвная реализация интерфейса:");

IInventoryOperations inventoryInterface = personal;
inventoryInterface.PrintSummary();

Склад 'Личный склад' добавлен в коллекцию.
Склад 'Склад электроники' добавлен в коллекцию.
Склад 'Личный склад' добавлен в коллекцию.
Склад 'Склад электроники' добавлен в коллекцию.
Склад 'Автоматический склад' добавлен в коллекцию.
Склад 'Умный склад' добавлен в коллекцию.

Отчеты по всем складам:
Склад 'Личный склад' (ID: 1): позиций 0, объем 0/500, открыт: True, владелец: Иван, заблокирован: False
Склад 'Склад электроники' (ID: 2): позиций 0, объем 0/1000, открыт: True, группа: Электроника, отдел: Общий, бюджет: 0
Склад 'Автоматический склад' (ID: 3): позиций 0, объем 0/2000, открыт: True, автоматизация: Высокий, робот: Стандартный, ПО: 1.0.0
Склад 'Умный склад' (ID: 4): позиций 0, объем 0/3000, открыт: True, автоматизация: Максимальный, робот: Стандартный, ПО: 1.0.0, климат-контроль: да, температура: 20°C

Количество складов: 4
[Склад 'Склад электроники'] Товар 'Ноутбук' успешно добавлен.
[Групповой склад] Товар 'Ноутбук' успешно отнесен к товарной группе 'Электроника'.

Поиск по н